
# UNSW-NB15: Loading Normal Traffic and DoS Attacks

This notebook prepares a compact binary anomaly-detection dataset from **UNSW-NB15**.

The complete UNSW-NB15 dataset contains approximately 2.54 million network-flow records and nine attack categories. **We do not download the complete dataset here.**

Instead, we use only the two smaller predefined partitions supplied for machine-learning experiments:

- `UNSW_NB15_training-set.csv`
- `UNSW_NB15_testing-set.csv`

Together, these contain about 258,000 rows.

We retain only:

- **Normal traffic**: `y = 0`
- **Denial-of-Service (DoS) traffic**: `y = 1`

Official dataset description and citation:

- [UNSW-NB15 official dataset page](https://research.unsw.edu.au/projects/unsw-nb15-dataset)
- Moustafa, N., & Slay, J. (2015). *UNSW-NB15: a comprehensive data set for network intrusion detection systems*. MilCIS 2015.

> The loader below downloads only the two predefined training and testing CSV files. It does not download the four large CSV files that make up the full 2.54-million-row dataset.



## What is a DoS attack?

A **Denial-of-Service attack** tries to make a server, network service, or communication resource unavailable to legitimate users. A common strategy is to overwhelm the target with unusually many connection attempts, packets, or requests.

Examples include:

- flooding a server with TCP connection requests;
- generating very high packet or byte rates;
- sending traffic for which the victim produces little useful response;
- keeping many connections incomplete or forcing them to reset.

In this notebook, each row represents an aggregated network flow rather than an individual packet. We therefore detect DoS behaviour through flow properties such as duration, packet counts, byte counts, traffic rate, protocol, service, and connection state.



## Selected features

We use 13 compact and interpretable features. They are deliberately chosen to describe:

- **what kind of traffic it is**;
- **how much traffic is generated**;
- **how fast it is generated**;
- **how the source and destination behave relative to each other**.

| Feature | Meaning | Expected relevance for DoS |
|---|---|---|
| `proto` | Network protocol, such as TCP, UDP, or ICMP | Different DoS mechanisms use different protocols |
| `service` | Application service, such as HTTP or DNS | Attacks often target particular services |
| `state` | Connection state | Flooding may produce incomplete or reset connections |
| `dur` | Flow duration | DoS flows may be extremely short or unusually long |
| `spkts` | Source-to-destination packet count | Floods can generate many outgoing packets |
| `dpkts` | Destination-to-source packet count | Helps describe response asymmetry |
| `sbytes` | Source-to-destination bytes | Measures traffic volume generated by the source |
| `dbytes` | Destination-to-source bytes | Victim response may be small relative to incoming traffic |
| `rate` | Overall packet rate | Direct indicator of high-rate flooding |
| `sload` | Source bits per second | Expected to be one of the strongest load indicators |
| `dload` | Destination bits per second | Captures response rate and asymmetry |
| `sinpkt` | Mean source packet inter-arrival time | Floods often have very small inter-arrival times |
| `dinpkt` | Mean destination packet inter-arrival time | Describes response timing |

We intentionally omit TTL and packet-loss features from the paper's basic set because they are more dependent on routing paths, operating systems, and the specific network environment. The selected features focus more directly on the traffic mechanism of a DoS attack.


## Starting point for exploration

Use the combined feature matrix `X` and labels `y`. For this exploration phase, both original source partitions are combined into one dataset.

The final export cell saves `content/normal_dos.csv`, containing the 13 selected features and `is_anomaly` (0 = normal, 1 = DoS). Preparation consists of filtering, feature selection and target creation; numeric values and categorical tokens are retained.


## Imports

In [12]:

# Standard libraries
from pathlib import Path
import urllib.request

# Data handling
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")



## Download only the two smaller predefined partitions

We download only:

- `UNSW_NB15_training-set.csv`
- `UNSW_NB15_testing-set.csv`

We do not download the four large CSV files of the complete 2.54-million-row dataset.


In [13]:

DATA_DIR = Path("./content/unsw_nb15")
DATA_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "train": "UNSW_NB15_training-set.csv",
    "test": "UNSW_NB15_testing-set.csv",
}

# Public copies of the two predefined UNSW-NB15 partitions.
# These paths point to files that are currently present in the repository.
URLS = {
    "train": (
        "https://raw.githubusercontent.com/Nir-J/ML-Projects/master/"
        "UNSW-Network_Packet_Classification/UNSW_NB15_training-set.csv"
    ),
    "test": (
        "https://raw.githubusercontent.com/Nir-J/ML-Projects/master/"
        "UNSW-Network_Packet_Classification/UNSW_NB15_testing-set.csv"
    ),
}


def download_file(url, destination):
    """Download one CSV file and fail with a useful message."""
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already available: {destination.name}")
        return

    print(f"Downloading {destination.name} ...")
    try:
        urllib.request.urlretrieve(url, destination)
    except Exception as exc:
        raise RuntimeError(
            f"Could not download {destination.name} from:\n{url}\n\n"
            "The repository location may have changed. "
            "Open the URL in a browser or upload the CSV manually to Colab."
        ) from exc

    if destination.stat().st_size == 0:
        raise RuntimeError(f"The downloaded file {destination.name} is empty.")

    print(
        f"Saved to {destination} "
        f"({destination.stat().st_size / 1_000_000:.1f} MB)"
    )


for split, filename in FILES.items():
    download_file(URLS[split], DATA_DIR / filename)


Saved to content/unsw_nb15/UNSW_NB15_training-set.csv (32.3 MB)
Saved to content/unsw_nb15/UNSW_NB15_testing-set.csv (15.4 MB)


In [14]:

train_raw = pd.read_csv(DATA_DIR / FILES["train"])
test_raw = pd.read_csv(DATA_DIR / FILES["test"])

print("Original training partition:", train_raw.shape)
print("Original testing partition: ", test_raw.shape)

display(train_raw.head())


Original training partition: (175341, 45)
Original testing partition:  (82332, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121,tcp,-,FIN,6,4,258,172,74.087,252,254,"14,158.942","8,495.365",0,0,24.296,8.375,30.178,11.831,255,621772692,2202533631,255,0.000,0.000,0.000,43,43,0,0,1,0,1,1,1,1,0,0,0,1,1,0,Normal,0
1,2,0.650,tcp,-,FIN,14,38,734,42014,78.473,62,252,"8,395.112","503,571.312",2,17,49.915,15.433,61.427,"1,387.778",255,1417884146,3077387971,255,0.000,0.000,0.000,52,1106,0,0,43,1,1,1,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623,tcp,-,FIN,8,16,364,13186,14.170,62,252,"1,572.272","60,929.230",1,6,231.876,102.737,"17,179.587","11,420.926",255,2116150707,2963114973,255,0.112,0.061,0.050,46,824,0,0,7,1,2,1,1,3,0,0,0,2,6,0,Normal,0
3,4,1.682,tcp,ftp,FIN,12,12,628,770,13.677,62,252,"2,740.179","3,358.622",1,3,152.877,90.236,259.080,"4,991.785",255,1107119177,1047442890,255,0.000,0.000,0.000,52,64,0,0,1,1,2,1,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449,tcp,-,FIN,10,6,534,268,33.374,254,252,"8,561.499","3,987.060",2,1,47.750,75.660,"2,415.838",115.807,255,2436137549,1977154190,255,0.128,0.071,0.057,53,45,0,0,43,1,2,2,1,40,0,0,0,2,39,0,Normal,0



## Check the schema

The columns `attack_cat` and `label` contain the targets supplied with the dataset:

- `attack_cat` gives the attack family, for example `DoS`.
- `label` is the original binary label: normal or any attack.

We will create our own binary target after filtering, because the notebook focuses specifically on **Normal versus DoS**.


In [15]:

required_columns = {
    "attack_cat", "label",
    "proto", "service", "state",
    "dur", "spkts", "dpkts",
    "sbytes", "dbytes",
    "rate", "sload", "dload",
    "sinpkt", "dinpkt",
}

for partition_name, partition in [("train", train_raw), ("test", test_raw)]:
    missing_columns = required_columns.difference(partition.columns)
    if missing_columns:
        raise ValueError(
            f"The {partition_name} partition does not match the expected schema. "
            f"Missing columns: {sorted(missing_columns)}"
        )

print(f"Schema check passed. Number of columns: {train_raw.shape[1]}")
print("\nAttack categories in the training partition:")
display(train_raw["attack_cat"].value_counts(dropna=False))


Schema check passed. Number of columns: 45

Attack categories in the training partition:


attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64


## Combine the original partitions and retain Normal and DoS

In these files, normal observations may appear as `"Normal"` in `attack_cat`. We standardize the text before filtering to avoid problems caused by capitalization or whitespace.


In [16]:

df_raw = pd.concat(
    [
        train_raw.assign(original_partition="train"),
        test_raw.assign(original_partition="test"),
    ],
    ignore_index=True,
)

df_raw["attack_cat_clean"] = (
    df_raw["attack_cat"]
    .fillna("Normal")
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Unique standardized attack categories:")
print(sorted(df_raw["attack_cat_clean"].unique()))


Unique standardized attack categories:
['analysis', 'backdoor', 'dos', 'exploits', 'fuzzers', 'generic', 'normal', 'reconnaissance', 'shellcode', 'worms']


In [17]:

selected_categories = ["normal", "dos"]

df = (
    df_raw.loc[df_raw["attack_cat_clean"].isin(selected_categories)]
    .copy()
    .reset_index(drop=True)
)

# Create the binary target used throughout the notebook:
# 0 = normal traffic, 1 = DoS attack
df["is_anomaly"] = (df["attack_cat_clean"] == "dos").astype(int)

print("Filtered dataset shape:", df.shape)
display(
    df["attack_cat_clean"]
    .value_counts()
    .rename_axis("class")
    .to_frame("count")
)


Filtered dataset shape: (109353, 48)


,count
class,
normal,93000
dos,16353



## Keep the compact feature set

categorical and numerical variables are kept separate.


In [18]:

CATEGORICAL_FEATURES = [
    "proto",
    "service",
    "state",
]

NUMERIC_FEATURES = [
    "dur",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
    "rate",
    "sload",
    "dload",
    "sinpkt",
    "dinpkt",
]

FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES
TARGET = "is_anomaly"

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nSelected features:")
print(FEATURES)

display(X.head())


X shape: (109353, 13)
y shape: (109353,)

Selected features:
['proto', 'service', 'state', 'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sinpkt', 'dinpkt']


,proto,service,state,dur,spkts,dpkts,sbytes,dbytes,rate,sload,dload,sinpkt,dinpkt
0,tcp,-,FIN,0.121,6,4,258,172,74.087,"14,158.942","8,495.365",24.296,8.375
1,tcp,-,FIN,0.650,14,38,734,42014,78.473,"8,395.112","503,571.312",49.915,15.433
2,tcp,-,FIN,1.623,8,16,364,13186,14.170,"1,572.272","60,929.230",231.876,102.737
3,tcp,ftp,FIN,1.682,12,12,628,770,13.677,"2,740.179","3,358.622",152.877,90.236
4,tcp,-,FIN,0.449,10,6,534,268,33.374,"8,561.499","3,987.060",47.750,75.660


## Save the combined dataset for exploration

Save all selected Normal/DoS flows to **`content/normal_dos.csv`**. The CSV contains exactly the 13 selected features plus `is_anomaly`, without a DataFrame index or partition column.


In [20]:
OUTPUT_PATH = Path("content/normal_dos.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

prepared = X.copy()
prepared[TARGET] = y
prepared.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(prepared):,} flows and {prepared.shape[1]} columns to {OUTPUT_PATH}")


Saved 109,353 flows and 14 columns to content/normal_dos.csv
